# 31_frozen_encoder_expert — Deeper-gate, gate-only Stage C

This is a clean trainability ablation of `31_deeper_C`. It uses the same dropout-free `64 → 32 → 4` ReLU gate and loads the exact Notebook 31 Stage-A encoder and Stage-B linear experts, but freezes both encoder and expert bank throughout Stage C. Only the 2,212 gate parameters train. The ordinary all-trainable deeper-gate run is included as the direct control.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('baseline_linear_gate', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('deeper_gate_all_trainable', 'config/compact_soft_moe_deeper_gate_3seed.yaml', 'nfv3_4way_moe_soft_deeper_gate32_v1'),
    ('deeper_gate_gate_only', 'config/compact_soft_moe_deeper_gate_frozen_stagec_3seed.yaml', 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_v1'),
]
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS_PER_STUDY = 3
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight, dry-run, and resumable execution

The gate-only challenger must schedule Stage C alone, reuse both signed baseline checkpoints, and retain the same topology and optimizer as the all-trainable deeper-gate control.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
frozen = yaml.safe_load(Path('config/compact_soft_moe_deeper_gate_frozen_stagec_3seed.yaml').read_text())
assert frozen['reuse_stage_a_from_prefix'] == frozen['reuse_stage_b_from_prefix'] == 'nfv3_4way_moe_soft_comparable_repro_v1'
assert frozen['backbone']['model']['gate']['hidden_dims'] == [32]
assert frozen['backbone']['model']['expert']['hidden_dims'] == []
assert frozen['backbone']['training']['stage_c_unfreeze'] == 'none'
assert frozen['backbone']['training']['stage_c']['freeze_experts'] is True
assert frozen['backbone']['training']['stage_c']['optimizer']['lr'] == 0.001
assert frozen['expected_total_parameters'] == 22332
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_global_residual_experts.py', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    print(f'=== {condition}: parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-frozen-{condition}-dry-run')
    if EXECUTE: run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-frozen-{condition}-training')
if not EXECUTE: print('NO TRAINING WAS STARTED. Review the contracts, set EXECUTE=True, and rerun this cell.')

## Paired validation, locked-test, per-class, and resource comparison

In [ ]:
import pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}; print('Completion:', complete)
if all(complete.values()):
    validation = []; test = []; classes = []; resources = []
    for name, path in summaries.items():
        for filename, target in [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Three_Seed_Resource_Accounting.csv', resources)]:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True); test = pd.concat(test, ignore_index=True)
    display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1')
    pivot['delta_gate_only_minus_all_trainable'] = pivot['deeper_gate_gate_only'] - pivot['deeper_gate_all_trainable']; display(pivot)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    resource_frame = pd.concat(resources, ignore_index=True); display(resource_frame)
    frozen_rows = resource_frame.query("condition == 'deeper_gate_gate_only'")
    assert set(frozen_rows['trainable_parameters'].astype(int)) == {2212}, 'Only the deeper gate should be trainable'
else: print('Comparison appears after all three three-seed manifests are complete.')

## Freeze-integrity and routing diagnostics

Because the encoder and experts are frozen, each Stage-C owned-expert score must exactly match its reused Stage-B score. Routing tables remain free to change because the gate is the only optimized module.

In [ ]:
if complete.get('deeper_gate_gate_only'):
    import numpy as np
    prefix = 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_v1'; baseline = 'nfv3_4way_moe_soft_comparable_repro_v1'
    preservation = []
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        stage_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{baseline}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
        stage_c = pd.read_csv(result_dir / 'Validation_Expert_Owned_StageC.csv')
        merged = stage_b.merge(stage_c, on='dataset', suffixes=('_stage_b', '_stage_c'))
        for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'): merged[f'delta_{metric}'] = merged[f'{metric}_stage_c'] - merged[f'{metric}_stage_b']
        assert np.allclose(merged[[f'delta_{metric}' for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy')]], 0.0, atol=1e-12)
        merged.insert(0, 'seed', seed); preservation.append(merged)
        print(f'=== Seed {seed}: gate by source dataset ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
        print(f'=== Seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))
    print('=== Exact Stage-B competence preservation ==='); display(pd.concat(preservation, ignore_index=True))

## Stage-C training and latent diagnostics

In [ ]:
if all(complete.values()):
    for name, path in summaries.items():
        print('===', name, 'Stage-C history ===')
        history = pd.read_csv(path / 'Training_History_3Seed.csv').query("stage == 'C'")
        display(history[['seed', 'epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left']])
        display(pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'))
        for figure in sorted((path / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))